In [0]:
from pyspark.sql.functions import (
    col, row_number, current_timestamp, sha2, concat_ws,
    expr, lit, lead
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
prices_bronze_df = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_prices_raw")

prices_cleaned_df = (prices_bronze_df
    .withColumnRenamed("series", "series_bk")
    .withColumnRenamed("product-name", "product_name")
    .withColumn("effective_from", expr("try_cast(period as date)"))
    .withColumn("price", expr("try_cast(value as decimal(10,3))"))
    .select(
        "series_bk", "product_name", "units", "price", "effective_from",
        "source_filename", "ingestion_timestamp"
    )
)

In [0]:
w_prices = Window.partitionBy("series_bk", "effective_from") \
                  .orderBy(col("ingestion_timestamp").desc())

prices_deduped_df = (prices_cleaned_df
    .withColumn("rn", row_number().over(w_prices))
    .filter(col("rn") == 1)
    .drop("rn")
)

prices_deduped_df.printSchema()
prices_deduped_df.count()

In [0]:
w_ordered = Window.partitionBy("series_bk").orderBy("effective_from")

prices_final_df = prices_deduped_df.withColumn(
    "price_sk",
    sha2(concat_ws("||", col("series_bk"), col("effective_from").cast("string")), 256)
).withColumn(
    "effective_to", lead("effective_from", 1).over(w_ordered)
).withColumn(
    "is_current", col("effective_to").isNull()
).withColumn(
    "_source_system", lit("EIA_petroleum_prices")
).withColumn(
    "_ingested_at", current_timestamp()
).select(
    "price_sk", "series_bk", "product_name", "units", "price",
    "effective_from", "effective_to", "is_current",
    "_source_system", "_ingested_at"
)

In [0]:
prices_final_df.groupBy("series_bk").agg(
    expr("sum(case when is_current then 1 else 0 end) as current_count")
).filter(col("current_count") != 1).count()

In [0]:
prices_final_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "false") \
    .saveAsTable("dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices")

In [0]:
%sql
SELECT is_current, COUNT(*) 
FROM dbr_dev_ua5816bd.roksolana_shendiu770_silver.petroleum_prices
GROUP BY is_current